<a href="https://colab.research.google.com/github/karlinnn/ML-Practice/blob/main/DT_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
 import numpy as np

class Node:
    def __init__(self,
                 feature=None,
                 threshold=None,
                 left=None,
                 right=None,
                 value=None):

        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


class DecisionTree:

    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.root = None

    # ------------------------
    # Entropy
    # ------------------------
    def entropy(self, y):

        classes, counts = np.unique(y, return_counts=True)

        probs = counts / len(y)

        entropy = -np.sum(probs * np.log2(probs))

        return entropy

    # ------------------------
    # Information Gain
    # ------------------------
    def information_gain(self,
                         parent,
                         left,
                         right):

        if len(left) == 0 or len(right) == 0:
            return 0

        parent_entropy = self.entropy(parent)

        left_weight = len(left) / len(parent)
        right_weight = len(right) / len(parent)

        child_entropy = (
            left_weight * self.entropy(left)
            +
            right_weight * self.entropy(right)
        )

        gain = parent_entropy - child_entropy

        return gain

    # ------------------------
    # Best Split
    # ------------------------
    def best_split(self, X, y):

        best_gain = -1

        best_feature = None
        best_threshold = None

        n_features = X.shape[1]

        print("\nTrying all possible splits...\n")

        for feature in range(n_features):

            thresholds = np.unique(
                X[:, feature]
            )

            for threshold in thresholds:

                left_mask = (
                    X[:, feature] <= threshold
                )

                right_mask = (
                    X[:, feature] > threshold
                )

                left_y = y[left_mask]
                right_y = y[right_mask]

                gain = self.information_gain(
                    y,
                    left_y,
                    right_y
                )

                print(
                    f"Feature {feature}, "
                    f"Threshold {threshold} "
                    f"=> IG = {gain:.4f}"
                )

                if gain > best_gain:

                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold

        print("\nBEST SPLIT FOUND")
        print(
            f"Feature {best_feature}, "
            f"Threshold {best_threshold}, "
            f"Gain = {best_gain:.4f}"
        )

        return best_feature, best_threshold

    # ------------------------
    # Majority Class
    # ------------------------
    def most_common_label(self, y):

        values, counts = np.unique(
            y,
            return_counts=True
        )

        return values[np.argmax(counts)]

    # ------------------------
    # Build Tree
    # ------------------------
    def build_tree(
        self,
        X,
        y,
        depth=0
    ):

        indent = "   " * depth

        print("\n")
        print("=" * 60)

        print(
            f"{indent}Building Node "
            f"(Depth={depth})"
        )

        print(
            f"{indent}Samples = {len(y)}"
        )

        print(
            f"{indent}Labels  = {y}"
        )

        node_entropy = self.entropy(y)

        print(
            f"{indent}Entropy = "
            f"{node_entropy:.4f}"
        )

        # stop conditions

        if (
            depth >= self.max_depth
            or len(np.unique(y)) == 1
            or len(y) < 2
        ):

            leaf_value = (
                self.most_common_label(y)
            )

            print(
                f"{indent}LEAF NODE "
                f"-> Class {leaf_value}"
            )

            return Node(
                value=leaf_value
            )

        feature, threshold = (
            self.best_split(X, y)
        )

        left_mask = (
            X[:, feature] <= threshold
        )

        right_mask = (
            X[:, feature] > threshold
        )

        print(
            f"\n{indent}Splitting:"
        )

        print(
            f"{indent}Feature {feature}"
        )

        print(
            f"{indent}Threshold "
            f"{threshold}"
        )

        print(
            f"{indent}Left Samples = "
            f"{np.sum(left_mask)}"
        )

        print(
            f"{indent}Right Samples = "
            f"{np.sum(right_mask)}"
        )

        left_child = self.build_tree(
            X[left_mask],
            y[left_mask],
            depth + 1
        )

        right_child = self.build_tree(
            X[right_mask],
            y[right_mask],
            depth + 1
        )

        return Node(
            feature=feature,
            threshold=threshold,
            left=left_child,
            right=right_child
        )

    # ------------------------
    # Fit
    # ------------------------
    def fit(self, X, y):

        self.root = self.build_tree(X, y)


# =====================================
# DATASET
# =====================================

# Feature 0 = CGPA
# Feature 1 = Attendance

X = np.array([
    [9,95],
    [8,90],
    [7,85],
    [6,70],
    [5,60]
])

y = np.array([
    1,
    1,
    1,
    0,
    0
])

# =====================================
# TRAIN
# =====================================

tree = DecisionTree(
    max_depth=3
)

tree.fit(X, y)



Building Node (Depth=0)
Samples = 5
Labels  = [1 1 1 0 0]
Entropy = 0.9710

Trying all possible splits...

Feature 0, Threshold 5 => IG = 0.3219
Feature 0, Threshold 6 => IG = 0.9710
Feature 0, Threshold 7 => IG = 0.4200
Feature 0, Threshold 8 => IG = 0.1710
Feature 0, Threshold 9 => IG = 0.0000
Feature 1, Threshold 60 => IG = 0.3219
Feature 1, Threshold 70 => IG = 0.9710
Feature 1, Threshold 85 => IG = 0.4200
Feature 1, Threshold 90 => IG = 0.1710
Feature 1, Threshold 95 => IG = 0.0000

BEST SPLIT FOUND
Feature 0, Threshold 6, Gain = 0.9710

Splitting:
Feature 0
Threshold 6
Left Samples = 2
Right Samples = 3


   Building Node (Depth=1)
   Samples = 2
   Labels  = [0 0]
   Entropy = -0.0000
   LEAF NODE -> Class 0


   Building Node (Depth=1)
   Samples = 3
   Labels  = [1 1 1]
   Entropy = -0.0000
   LEAF NODE -> Class 1
